# Deep learning on images

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [ ]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model, get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-02 14:22:16.318101: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-02 14:22:16.359679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-02 14:22:17.305219: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.image)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)

<module 'src.models.on_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

In [9]:
# Class distribution
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [ ]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = False  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if rebalance_with_weights:
    if small_train_sample:
        raise ValueError("Warning: rebalance_with_weights should not be used with small_train_sample.")
    print('using class weights')
else:
    print('not using class weights')

not using class weights


In [13]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [14]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024587
min       0.009028
25%       0.018449
50%       0.031158
75%       0.055741
max       0.118646
Name: proportion, dtype: float64

In [15]:
y_train.value_counts().describe()

count      27.000000
mean      754.814815
std       501.078609
min       184.000000
25%       376.000000
50%       635.000000
75%      1136.000000
max      2418.000000
Name: count, dtype: float64

In [16]:
print(X_train.shape)

(20380, 31)


In [17]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [ ]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

I0000 00:00:1759407739.151469   17608 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [ ]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Create model

In [23]:
# started_from_subversion=None
# model = define_model(embedding_dim=16, n_cols_tabular=n_cols_tabular, num_classes=27)

### Load a saved model

In [24]:
subversion_to_load=1
while True:
    location_to_load = Path(artifacts_folder / f'best_model-{subversion_to_load}.keras')
    if not location_to_load.exists():
        subversion_to_load-=1
        break
    subversion_to_load+=1
subversion_to_load

3

In [ ]:
started_from_subversion=subversion_to_load
model = keras.models.load_model(artifacts_folder / f'best_model-{started_from_subversion}.keras')

### Load saved weights

In [26]:
# started_from_subversion=1
# model.load_weights(artifacts_folder / f'best_model-{started_from_subversion}.h5')

### Summary

In [27]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 500, 500,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 500, 500,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 500, 500,  │          0 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 250, 250,  │        864 │ normalization[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 250, 250,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 250, 250,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 250, 250,  │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 250, 250,  │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 250, 250,  │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 125, 125,  │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 125, 125,  │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 125, 125,  │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 125, 125,  │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 125, 125,  │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 125, 125,  │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 125, 125,  │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 125, 125,  │          0 │ block2b_expand_b

 Total params: 12,654,275 (48.27 MB)

 Trainable params: 2,244,987 (8.56 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

 Optimizer params: 4,489,976 (17.13 MB)

## Training

### Callbacks

#### Saving

In [28]:
# Pick an available filename to save a model.
subversion=1
while True:
    new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
    if not new_location_for_saving_model.exists():
        break
    subversion+=1
# new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
new_location_for_saving_model


PosixPath('artifacts/on_images/deep_learning/v1/best_model-4.keras')

In [ ]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

In [30]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

#### EarlyStopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

#### ReduceLROnPlateau

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

#### TerminateOnNaN

In [33]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

#### TensorBoard

In [34]:
tensor_board_folder = artifacts_folder / "tensor_board"
tensor_board = callbacks.TensorBoard(log_dir = tensor_board_folder)

### Configuration 2

#### max_epochs

In [35]:
# max_epochs=4  # TODO: try 50

In [36]:
# Pick max_epochs based on your available time
available_minutes=50

import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793
max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

8

#### misc

In [37]:
learning_rate=0.001

In [38]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

### Training

In [41]:
raise Exception(f"Are you sure you want to launch training with {max_epochs=} and {started_from_subversion=} ({available_minutes=}) ?")

Exception: Are you sure you want to launch training with max_epochs=8 and started_from_subversion=3 (available_minutes=50) ?

In [42]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=max_epochs, callbacks=callbacks, verbose=1)
end = time.time()
total_minutes = (end - start)/60
total_minutes

Epoch 1/8


2025-10-02 14:23:17.475866: I external/local_xla/xla/service/service.cc:163] XLA service 0x7a7668014e50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-02 14:23:17.475901: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-02 14:23:17.733078: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-02 14:23:19.080573: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-02 14:23:19.946750: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-02 14:23:19.

636/637 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.7507 - loss: 0.8612

2025-10-02 14:25:03.913061: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-02 14:25:13.185867: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 14:25:13.285863: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 14:25:14.214833: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

637/637 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.7506 - loss: 0.8616

2025-10-02 14:26:31.525469: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-02 14:26:39.049764: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 14:26:39.148359: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-02 14:26:39.986689: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

637/637 ━━━━━━━━━━━━━━━━━━━━ 215s 291ms/step - accuracy: 0.6604 - loss: 1.1549 - val_accuracy: 0.6082 - val_loss: 1.3050 - learning_rate: 0.0010
Epoch 2/8
637/637 ━━━━━━━━━━━━━━━━━━━━ 141s 221ms/step - accuracy: 0.7979 - loss: 0.7176 - val_accuracy: 0.5419 - val_loss: 1.6616 - learning_rate: 0.0010
Epoch 3/8
637/637 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.9530 - loss: 0.2064
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
637/637 ━━━━━━━━━━━━━━━━━━━━ 141s 221ms/step - accuracy: 0.9444 - loss: 0.2392 - val_accuracy: 0.4835 - val_loss: 2.2312 - learning_rate: 0.0010
Epoch 4/8
637/637 ━━━━━━━━━━━━━━━━━━━━ 140s 220ms/step - accuracy: 0.9844 - loss: 0.0775 - val_accuracy: 0.5015 - val_loss: 2.2039 - learning_rate: 2.0000e-04


10.631144014994304

## Evaluation

In [43]:
# To use tensorboard:
# open a terminal from the root of the repo and run these two commands (the path must match the value of tensor_board_folder).
# source venv/bin/activate
# tensorboard --logdir artifacts/on_images/deep_learning/v1/tensor_board
tensor_board_folder

PosixPath('artifacts/on_images/deep_learning/v1/tensor_board')

In [ ]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.6082195043563843,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576}

In [45]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

531/531 ━━━━━━━━━━━━━━━━━━━━ 72s 126ms/step


array([[1.4514719e-04, 1.7099828e-03, 2.5714685e-03, ..., 4.2570787e-03,
        1.7613146e-04, 1.9240444e-05],
       [1.2029868e-03, 5.0938614e-03, 4.9275369e-03, ..., 3.1577837e-02,
        1.6205997e-05, 2.8383691e-05],
       [4.9220043e-06, 4.7372556e-03, 1.6641943e-02, ..., 5.8717233e-01,
        2.7251451e-06, 2.2112730e-05],
       ...,
       [3.4320053e-01, 8.6691668e-03, 4.0342093e-05, ..., 1.1236330e-05,
        4.3745181e-01, 4.0550378e-05],
       [4.8362520e-03, 1.5600659e-02, 1.8950639e-02, ..., 1.0683927e-01,
        9.1652351e-04, 9.0620597e-04],
       [9.3750749e-03, 1.4033916e-02, 3.4119494e-02, ..., 1.0876027e-01,
        4.3227954e-04, 1.3099304e-05]], shape=(16984, 27), dtype=float32)

In [46]:
from sklearn import metrics

In [47]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [48]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1180,1280,1281,1300,1301,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,,,
10,356,22,0,0,1,10,2,4,3,3,0,0,1,0,4,1,0,0,141,52,0,8,0,0,1,14,0
40,18,260,5,2,9,37,1,6,3,60,0,2,2,1,2,1,7,0,43,7,10,5,0,1,15,3,2
50,1,13,75,10,17,9,0,13,0,95,0,12,0,4,0,0,9,0,2,12,6,13,0,15,30,0,0
60,0,11,7,88,0,3,0,5,0,41,0,0,0,0,0,0,0,0,1,0,5,4,0,1,0,0,0
1140,4,15,2,0,385,13,10,31,4,19,1,3,3,1,4,2,6,0,12,6,0,5,0,4,3,0,1
1160,4,16,0,0,8,706,0,1,4,4,0,0,0,0,0,0,0,0,29,13,3,2,0,0,1,0,0
1180,10,5,0,0,32,8,28,12,4,8,1,0,2,2,1,0,3,0,14,9,1,2,0,3,5,2,1
1280,4,9,3,1,100,4,9,349,22,227,2,53,9,20,12,0,53,2,7,8,2,13,0,11,52,1,1
1281,21,28,1,0,13,23,3,100,79,24,0,11,4,3,1,2,18,0,15,20,2,18,0,5,17,2,4


In [49]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.5981004513860582

In [50]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

,precision,recall,f1-score,support
10,0.496513,0.571429,0.531343,623.00000
40,0.523139,0.517928,0.520521,502.00000
50,0.625000,0.223214,0.328947,336.00000
60,0.854369,0.530120,0.654275,166.00000
1140,0.578947,0.720974,0.642202,534.00000
1160,0.805936,0.892541,0.847031,791.00000
1180,0.417910,0.183007,0.254545,153.00000
1280,0.420482,0.358316,0.386918,974.00000
1281,0.519737,0.190821,0.279152,414.00000
1300,0.496956,0.889990,0.637784,1009.00000


In [51]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.632264,0.537501,0.553020,629.037037
std,0.163352,0.232970,0.180580,420.759339
min,0.262990,0.183007,0.254545,153.000000
25%,0.508936,0.348172,0.391118,310.000000
50%,0.625000,0.534535,0.579853,534.000000
75%,0.774968,0.708313,0.662677,953.500000
max,0.884793,0.892541,0.847031,2042.000000


In [52]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.293638,0.607203,-0.006602
recall,0.293638,1.000000,0.917933,0.080669
f1-score,0.607203,0.917933,1.000000,0.065549
support,-0.006602,0.080669,0.065549,1.000000


In [53]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.5981004513860582

In [54]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.6082195043563843,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576,
 'weighted_avg_f1_score': 0.5981004513860582,
 'min_f1_score': np.float64(0.2545454545454545),
 'std_f1_score': np.float64(0.1805798797538243)}

## Update tracker

In [55]:
to_track=['subversion','started_from_subversion','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model)
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.6082195043563843,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576,
 'weighted_avg_f1_score': 0.5981004513860582,
 'min_f1_score': np.float64(0.2545454545454545),
 'std_f1_score': np.float64(0.1805798797538243),
 'subversion': 4,
 'started_from_subversion': 3,
 'max_epochs': 8,
 'rebalance_with_weights': False,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [ ]:
tracker['comment']="Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers."

In [57]:
tracker

{'X_train.shape[0]': 20380,
 'val_accuracy': 0.6082195043563843,
 'actual_epochs': 4,
 'minutes_per_epoch': 2.657786003748576,
 'weighted_avg_f1_score': 0.5981004513860582,
 'min_f1_score': np.float64(0.2545454545454545),
 'std_f1_score': np.float64(0.1805798797538243),
 'subversion': 4,
 'started_from_subversion': 3,
 'max_epochs': 8,
 'rebalance_with_weights': False,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16},
 'comment': 'Increased frac from 0.1 to 0.3 for small train sample size.'}

In [58]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [59]:
log_file_path=artifacts_folder / 'experiments.parquet'
log_file_path

PosixPath('artifacts/on_images/deep_learning/v1/experiments.parquet')

In [60]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, log_file_path=log_file_path)

Expérience sauvegardée dans artifacts/on_images/deep_learning/v1/experiments.parquet.


## Show tracking logs

In [61]:
log_file_path=artifacts_folder / 'experiments.parquet'
log_file_path

PosixPath('artifacts/on_images/deep_learning/v1/experiments.parquet')

In [62]:
pd.set_option('max_colwidth', None)

In [63]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
